In [9]:
import re

# Parsing log files
This is an example of a typical work of Sys Admins. The file in this example is an extract of a typical linux log file.

In [4]:
with open("logfile.txt","r") as f:
    lines = f.readlines() 

In [6]:
print(lines[0])
print(lines[1])

2019-11-05 14:25:07 status half-configured python3-qrcode:all 5.3-1

2019-11-05 14:25:09 status installed python3-qrcode:all 5.3-1



Let's assume we are interested in parsing the log file. All lines are of the form `[date] [time] <message>` we want to create a dictionary of the form:
```
{ "date" : {
        "time" : ["message", ...],
        ...
        },
   ...
}
```
That is we want to create a data structure that for each day, organized by time returns the list of the messages.  
e.g.:
```
mydict[date][time] can give [ "msg1", "msg2", "msg3" , ... ]
```
We can use Regular Expressions to parse the string of the log file and extract the date and time.  
Regular expressions are a powerful language to express search and matching of patterns of strings. Python's `re` package provides functionality similar to regular expressions from `perl`, probably the most powerful support available for RE.  
Full documentation of python's RE support is available [here](https://docs.python.org/3/library/re.html).

In [10]:
#A simple expression that matches "abc" "abcd" but not "dd"
simple_pattern = r".*abc.*"
re.search(simple_pattern, "abc")

<re.Match object; span=(0, 3), match='abc'>

In [11]:
re.search(simple_pattern, "abcd")

<re.Match object; span=(0, 4), match='abcd'>

In [12]:
re.search(simple_pattern, "dd")

The string `.*` means: match any character `.` (the symbol dot means any character), any number of times from 0 to infinity (`*`). Thus `.*abc.*` matches any number of characters, followed by the string `abc` followed by any other number of characters. 
For this exercise we need other useful expressions:

 * `^` : Matches the beginning of the string
 * `$` : Matches the end of the string
 * `[0-9]` : Matches any single digit
 * `{2,4}` : Matches a minimum of 2 a maximum of 4 of the *preceding* expression. I.e.: `[0-9]{2,4}` matches 2, 3 or 4 digits number. `{4}` matches exactly four preceding expressions.
 * `\s` : Matches any white space
 * `*` : Matches any number of the previous expression
 * `+` : Matches at least one occurrence of the previous expression
 

In [14]:
simple_pattern = r"^[0-9]{4}$"
re.search(simple_pattern, "1234")

<re.Match object; span=(0, 4), match='1234'>

In [16]:
re.search(simple_pattern, "12345") #Too many chars

A date is matched: `[0-9]{4}-[0-9]{2}-[0-9]{2}` e.g. `YYYY-MM-DD` 

Thus the full line can be matched by 4 digits, the character `-` two digits, `-`, two digits, a space, the time ("HH:MM:SS") at least a whitespace, a string:

In [30]:
line_pattern = r"^[0-9]{4}-[0-9]{2}-[0-9]{2}\s[0-9]{2}:[0-9]{2}:[0-9]{2}\s+.*$"

In [35]:
print(lines[0])
match = re.search(line_pattern, lines[0])
match

2019-11-05 14:25:07 status half-configured python3-qrcode:all 5.3-1



<re.Match object; span=(0, 67), match='2019-11-05 14:25:07 status half-configured python>

The RE we have created matches the example line, we now want to extract the date, time and message. We can do that defining groups in the regular expressions. Note the `(..)` that defines a group around a RE pattern.

In [36]:
line_pattern = r"^([0-9]{4}-[0-9]{2}-[0-9]{2})\s([0-9]{2}:[0-9]{2}:[0-9]{2})\s+(.*)$"

In [37]:
match = re.search(line_pattern, lines[0])
match.groups()

('2019-11-05', '14:25:07', 'status half-configured python3-qrcode:all 5.3-1')

We can now loop on all lines and create the data structure:

In [39]:
mydict = dict()
for line in lines:
    match = re.search(line_pattern, line)
    if match: #in case a line does not match the expected format
        date, time, message = match.groups()
        if date in mydict:
            if time in mydict[date]:
                mydict[date][time].append(message)
            else:
                mydict[date][time] = [ message ]
        else:
            mydict[date] = { time: [message] }


In [41]:
mydict

{'2019-11-05': {'14:25:07': ['status half-configured python3-qrcode:all 5.3-1'],
  '14:25:09': ['status installed python3-qrcode:all 5.3-1',
   'configure python3-wsaccel:amd64 0.6.2-1 <none>',
   'status unpacked python3-wsaccel:amd64 0.6.2-1',
   'status half-configured python3-wsaccel:amd64 0.6.2-1',
   'status installed python3-wsaccel:amd64 0.6.2-1',
   'configure libglew-dev:amd64 2.0.0-5 <none>',
   'status unpacked libglew-dev:amd64 2.0.0-5',
   'status half-configured libglew-dev:amd64 2.0.0-5',
   'status installed libglew-dev:amd64 2.0.0-5',
   'configure tcl-vtk7:amd64 7.1.1+dfsg1-2 <none>',
   'status unpacked tcl-vtk7:amd64 7.1.1+dfsg1-2',
   'status half-configured tcl-vtk7:amd64 7.1.1+dfsg1-2',
   'status installed tcl-vtk7:amd64 7.1.1+dfsg1-2',
   'configure python3-trie:all 0.2+ds-1 <none>',
   'status unpacked python3-trie:all 0.2+ds-1',
   'status half-configured python3-trie:all 0.2+ds-1'],
  '14:25:10': ['status installed python3-trie:all 0.2+ds-1',
   'configure 